# LC 56 — Merge Intervals
**Difficulty:** Medium &nbsp;|&nbsp; **Category:** Intervals
**Pattern:** Sort by Start — Extend Running End

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Sort by start time.
Walk through intervals keeping a running end. If
the next interval starts at or before the running
end, extend end. Otherwise seal the current merged
interval and start a new one.
</div>

## Official Problem Statement

Given an array of `intervals` where
`intervals[i] = [starti, endi]`, merge all
overlapping intervals, and return an array of the
non-overlapping intervals that cover all the
intervals in the input.

**Example 1:**
```
Input:  intervals = [[1,3],[2,6],[8,10],[15,18]]
Output: [[1,6],[8,10],[15,18]]
Explanation: [1,3] and [2,6] overlap -> merge to [1,6]
```
**Example 2:**
```
Input:  intervals = [[1,4],[4,5]]
Output: [[1,5]]
Explanation: [1,4] and [4,5] are adjacent -> merge.
```

**Constraints:**
- `1 <= intervals.length <= 10^4`
- `intervals[i].length == 2`
- `0 <= starti <= endi <= 10^4`

## What This Is Actually Asking

You have a list of time ranges. Some ranges overlap
or touch each other. Combine all overlapping ranges
into single bigger ranges. Return the cleaned-up
list with no overlaps.

## Walk Through an Example by Hand

```
intervals = [[1,3],[2,6],[8,10],[15,18]]

Step 1 — sort by start:
  [[1,3],[2,6],[8,10],[15,18]]  (already sorted)

Step 2 — walk and merge:
  result = []
  cur = [1, 3]

  [2,6]:   2 <= 3 (overlaps) -> extend end  cur=[1,6]
  [8,10]:  8 > 6  (gap)      -> seal [1,6]  cur=[8,10]
  [15,18]: 15 > 10 (gap)     -> seal [8,10] cur=[15,18]

  End of loop -> seal [15,18]

result = [[1,6],[8,10],[15,18]]

Overlap check: next_start <= cur_end
  [2,6] vs cur_end=3:  2 <= 3  YES overlap
  new end = max(3, 6) = 6
```

## The Picture

```
Before merge:
  [1----3]
     [2------6]
                 [8--10]
                              [15---18]

After sort by start:
  [1, 3]  [2, 6]  [8,10]  [15,18]

Walk left to right, track running end:

  cur_end = 3
  next starts at 2 — 2 <= 3 — OVERLAP
  extend: cur_end = max(3, 6) = 6

  cur_end = 6
  next starts at 8 — 8 > 6 — GAP
  seal [1,6], start new [8,10]

  cur_end = 10
  next starts at 15 — 15 > 10 — GAP
  seal [8,10], start new [15,18]

After merge:
  [1---------6]
                 [8--10]
                              [15---18]

Key rule: overlap iff next_start <= cur_end
```

## When To Use This Pattern

- When asked to **merge overlapping intervals**,
  think **sort by start, track running end**
- When `next_start <= cur_end`, think
  **overlap — extend end to max(cur_end, next_end)**
- When `next_start > cur_end`, think
  **gap — seal current, start new**
- When intervals may be unsorted on input, think
  **always sort by start first**

## The Approach

Sort the intervals by start time. Set the current
merged interval to the first one. Walk through
the remaining intervals. If the next interval's
start is within the current merged range, extend
the end to the maximum of both ends. Otherwise
add the current merged interval to the result and
start a new one. Append the last merged interval
when the loop ends.

In [ ]:
from typing import List  # type hints for the solution

In [ ]:
def test_harness(func):
    tests = [
        # (intervals, expected)
        ([[1,3],[2,6],[8,10],[15,18]], [[1,6],[8,10],[15,18]]),
        ([[1,4],[4,5]],               [[1,5]]),  # touching
        ([[1,4]],                     [[1,4]]),  # single
        ([[1,4],[2,3]],               [[1,4]]),  # contained
        ([[1,2],[3,4],[5,6]],         [[1,2],[3,4],[5,6]]),
        ([[1,10],[2,3],[4,5]],        [[1,10]]),  # all inside
        ([[2,3],[1,4]],               [[1,4]]),  # unsorted
        ([[1,3],[2,4],[3,5]],         [[1,5]]),  # chain
    ]

    passed = 0
    for i, (intervals, expected) in enumerate(tests):
        result = func([x[:] for x in intervals])
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(
            f"Test {i+1}: {status} | "
            f"intervals={intervals} | "
            f"expected={expected} | got={result}"
        )

    print(f"\n{passed}/{len(tests)} tests passed")

In [ ]:
def merge(
    intervals: List[List[int]]
) -> List[List[int]]:
    """
    Merge all overlapping intervals and return result.

    Sort by start. Walk: if next_start <= cur_end,
    extend cur_end to max(cur_end, next_end). Else
    seal cur and start new. Append last at end.

    Time:  O(n log n) — dominated by sort
    Space: O(n) — output list holds at most n intervals
    """
    pass


# Quick debug — run this cell while building
print(merge([[1,3],[2,6],[8,10],[15,18]]))
# [[1,6],[8,10],[15,18]]
print(merge([[1,4],[4,5]]))   # [[1,5]]
print(merge([[1,4],[2,3]]))   # [[1,4]]
print(merge([[2,3],[1,4]]))   # [[1,4]]

In [ ]:
# Uncomment and run when solution is ready
# test_harness(merge)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Brute force — check all pairs | O(n²) | O(n) |
| Sort + linear scan | O(n log n) | O(n) |

Sorting is the bottleneck. The merge walk itself
is O(n) — each interval is visited exactly once.

## Real World Connection

At Citi, the ETL scheduling system stores job
execution windows as [start, end] intervals. Before
launching a new batch, the system merges all
overlapping windows to find the true busy periods
on each server and avoid double-booking compute.
Merge Intervals is the exact algorithm run against
the telemetry scheduling table every morning.
On AWS Glue, partition time ranges returned by the
metadata catalog are merged before the job launcher
allocates DPUs — preventing redundant parallel
reads of the same S3 partition.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra